In [0]:
import dlt as dp
from pyspark.sql.functions import col, coalesce, lit, trim, when

def clean_col(c, default):
    """Replace NULL, empty string, or literal '' with a default value."""
    return (
        when(col(c).isNull() | (trim(col(c)) == "") | (col(c) == "''"), lit(default))
        .otherwise(col(c))
    )

@dp.table(
    name="dbr_dev.skybound_gold.gold_active_flights",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.zOrderCols": "icao24",
    },
)
def gold_active_flights():
    flights = (
        spark.read.table("dbr_dev.skybound_silver.flights_silver")
        .select(
            col("icao24"),
            when(trim(col("callsign")) == "", "N/A")
                .otherwise(coalesce(trim(col("callsign")), lit("N/A")))
                .alias("callsign"),
            col("origin_country"),
            col("latitude"),
            col("longitude"),
            coalesce(col("altitude"), lit(0.0)).alias("altitude"),
            coalesce(col("speed"), lit(0.0)).alias("speed"),
            coalesce(col("heading"), lit(0.0)).alias("heading"),
            coalesce(col("on_ground"), lit(False)).alias("on_ground"),
            col("last_seen")
        )
    )

    aircrafts = spark.read.table("dbr_dev.skybound_silver.aircrafts_silver")

    return (
        flights.join(aircrafts, "icao24", "left")
        .select(
            flights["*"],
            clean_col("registration", "Unknown").alias("registration"),
            clean_col("manufacturerName", "Private").alias("manufacturer_name"),
            clean_col("model", "Unknown Model").alias("model"),
            clean_col("operator", "Unknown Op").alias("operator"),
            clean_col("country", "Unknown").alias("aircraft_country")
        )
    )